# Case study: calibrating an SIR epidemic model with Tangent

The SIR model steps a population through **S**usceptible → **I**nfected →
**R**ecovered compartments. Fitting it to observed case counts means minimizing
a loss over the transmission rate `beta` and recovery rate `gamma` — a gradient
problem. Real models also carry **data-dependent control flow**: here a
"lockdown" cuts transmission *while infections exceed a threshold*, an `if`
inside the time-stepping loop.

That branch is exactly what makes hand-derived or graph-traced gradients
awkward. **Tangent** differentiates the loop *including the branch* as ordinary
Python, so we can calibrate the model by gradient descent.


In [ ]:
# Colab setup: install Tangent from PyPI (distributed as `tangent-ad`,
# imported as `tangent`). A no-op if it is already installed.
!pip install -q tangent-ad

In [1]:
import numpy as np
import tangent


## The simulator (plain NumPy, with an intervention branch)

Forward-Euler SIR. The `if I > threshold` branch models a lockdown that scales
transmission down while infections are high. Tangent's reverse mode transforms
the branch, so its effect on the gradient is accounted for automatically.

In [2]:
def sir_sse(beta, gamma, S0, I0, Npop, dt, n_steps, observed, threshold, reduction):
    """Sum of squared errors between simulated and observed infections."""
    S = S0
    I = I0
    loss = 0.0
    for t in range(n_steps):
        if I > threshold:                 # <-- data-dependent lockdown
            beta_eff = beta * reduction
        else:
            beta_eff = beta
        new_infections = beta_eff * S * I / Npop
        new_recoveries = gamma * I
        S = S - new_infections * dt
        I = I + (new_infections - new_recoveries) * dt
        diff = I - observed[t]
        loss = loss + diff * diff
    return loss


## A synthetic outbreak to fit

We generate "observed" infections from known true parameters, then pretend we
only see the curve and must recover `beta` and `gamma`.

In [3]:
Npop, dt, n_steps = 1000.0, 0.5, 60
S0, I0 = 999.0, 1.0
threshold, reduction = 60.0, 0.5
true_beta, true_gamma = 0.9, 0.2

def simulate_infections(beta, gamma):
    S, I, out = S0, I0, []
    for t in range(n_steps):
        beta_eff = beta * reduction if I > threshold else beta
        ni = beta_eff * S * I / Npop
        nr = gamma * I
        S = S - ni * dt
        I = I + (ni - nr) * dt
        out.append(I)
    return np.array(out)

observed = simulate_infections(true_beta, true_gamma)
CONSTS = (S0, I0, Npop, dt, n_steps, observed, threshold, reduction)

beta, gamma = 0.6, 0.3   # initial guess
print("peak observed infections: %.1f" % observed.max())
print("SSE at initial guess: %.1f" % sir_sse(beta, gamma, *CONSTS))


peak observed infections: 220.2
SSE at initial guess: 619392.1


## The gradient — differentiated through the branch

In [4]:
grad_sse = tangent.grad(sir_sse, wrt=(0, 1))
g_beta, g_gamma = grad_sse(beta, gamma, *CONSTS)
print("dSSE/dbeta  = %.2f" % g_beta)
print("dSSE/dgamma = %.2f" % g_gamma)

h = 1e-6
fd_b = (sir_sse(beta + h, gamma, *CONSTS) - sir_sse(beta - h, gamma, *CONSTS)) / (2 * h)
fd_g = (sir_sse(beta, gamma + h, *CONSTS) - sir_sse(beta, gamma - h, *CONSTS)) / (2 * h)
print("FD dbeta    = %.2f" % fd_b)
print("FD dgamma   = %.2f" % fd_g)
print("relative error: %.2e" % (max(abs(g_beta - fd_b), abs(g_gamma - fd_g)) / abs(g_beta)))


dSSE/dbeta  = -4907850.63
dSSE/dgamma = 6421770.04
FD dbeta    = -4907850.63
FD dgamma   = 6421770.04
relative error: 8.40e-11


The AD gradient matches finite differences even though the loop takes a
data-dependent branch each step — Tangent transformed the `if` correctly.

## Calibrate by gradient descent

A small Adam optimizer (robust to the very different scales of the two
gradients) recovers the parameters from the observed curve.

In [5]:
def adam_calibrate(beta, gamma, iters=600, lr=0.01):
    mb = mg = vb = vg = 0.0
    b1, b2, eps = 0.9, 0.999, 1e-8
    for it in range(1, iters + 1):
        gb, gg = grad_sse(beta, gamma, *CONSTS)
        mb = b1 * mb + (1 - b1) * gb; vb = b2 * vb + (1 - b2) * gb * gb
        mg = b1 * mg + (1 - b1) * gg; vg = b2 * vg + (1 - b2) * gg * gg
        beta -= lr * (mb / (1 - b1 ** it)) / (np.sqrt(vb / (1 - b2 ** it)) + eps)
        gamma -= lr * (mg / (1 - b1 ** it)) / (np.sqrt(vg / (1 - b2 ** it)) + eps)
    return beta, gamma

fit_beta, fit_gamma = adam_calibrate(0.6, 0.3)
print("recovered beta  = %.4f   (true %.2f)" % (fit_beta, true_beta))
print("recovered gamma = %.4f   (true %.2f)" % (fit_gamma, true_gamma))
print("SSE: %.1f -> %.2f" % (sir_sse(0.6, 0.3, *CONSTS), sir_sse(fit_beta, fit_gamma, *CONSTS)))


recovered beta  = 0.8735   (true 0.90)
recovered gamma = 0.1971   (true 0.20)
SSE: 619392.1 -> 271.37


## The gradient is readable Python

The generated adjoint contains the reverse of the lockdown branch — you can
read exactly how the gradient flows through it.

In [6]:
src = grad_sse.__tangent_source__
print("\n".join(src.splitlines()[:24]))
print("...  (%d lines total)" % len(src.splitlines()))


def dsir_ssedbetagamma(beta, gamma, S0, I0, Npop, dt, n_steps, observed, threshold, reduction, bloss=1.0):
    # Initialize the tape
    _stack = tangent.Stack()
    diff_times_diff = None
    diff = None
    _I = None
    _I2 = None
    _S = None
    new_recoveries = None
    new_infections = None
    _new_infections = None
    _new_infections2 = None
    beta_eff = None
    # Beginning of forward pass
    'Sum of squared errors between simulated and observed infections.'
    S = S0
    I = I0
    loss = 0.0
    i = 0
    for t in range(n_steps):
        cond = I > threshold
        if cond:
            tangent.push(_stack, beta_eff, '_5c7b434b')
            beta_eff = beta * reduction
...  (184 lines total)


## Takeaways

- An epidemic simulator with a **data-dependent intervention branch** was
  differentiated as ordinary Python — the `if` is handled by reverse mode, no
  smoothing or reformulation needed.
- The AD gradient matches finite differences, and drives a gradient-based
  **calibration** that recovers the transmission and recovery rates from a case
  curve.
- The adjoint is legible source you can inspect (`tangent.source_map`,
  `tangent.explain`), which is Tangent's edge over tracing frameworks for messy,
  branch-heavy scientific code.
